In [1]:
import pandas as pd
import numpy as np
import math

In [2]:
data = pd.read_csv("cfbweek15.csv")

In [3]:
priors = pd.read_csv("MasseyRatings_2024.csv")  # Team, MasseyRating
prior_dict = dict(zip(priors['Team'], priors['MasseyRating']))

In [5]:
filtered_data = data[
    (data["Completed"] == True)
    & (data["Week"] <= 16)
    & (data["SeasonType"] != 'postseason')
    & data["HomePoints"].notna()
    & data["AwayPoints"].notna()
]


# Extract necessary columns
games = filtered_data[['HomeTeam', 'AwayTeam', 'HomePoints', 'AwayPoints']]

# Calculate score differential
games['Score Differential'] = abs(games['HomePoints'] - games['AwayPoints'])

C:\Users\tanne\AppData\Local\Temp\ipykernel_31196\2063109948.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  games['Score Differential'] = abs(games['HomePoints'] - games['AwayPoints'])


In [6]:
import numpy as np
import pandas as pd

# Unique list of teams
teams = set(games['HomeTeam']).union(set(games['AwayTeam']))
teams_list = sorted(list(teams))  # sorted for consistency
team_to_idx = {team: i for i, team in enumerate(teams_list)}

In [20]:
# Build a list of equations for each game
# We'll end up with #games equations (+ priors + 1 sum-constraint), and we have #teams unknowns.

num_games = len(games)
n = len(teams_list)
winners_bonus = 2.75

# Encode function so changing it once applies to BOTH real games and priors
def encode_mov(mov: float, method = 'sqrt_with_bonus', cap = 35, winners_bonus = 2.75) -> float:
    if method == 'sqrt_with_bonus':
        target = np.sign(mov) * np.sqrt(abs(mov)) + np.sign(mov) * winners_bonus
    elif method == 'raw_dif':
        target = mov
    elif method == 'capped_with_bonus':
        target = min(mov, cap) + np.sign(mov) * winners_bonus
    return target

# --- Count games per team ---
games_played = {t: 0 for t in teams_list}
for _, row in games.iterrows():
    games_played[row['HomeTeam']] += 1
    games_played[row['AwayTeam']] += 1

# --- Effective games (real + prior) for normalization ---
prior_weight = .00005       # 1.0 ~ one game; increase to make priors fade more slowly
prior_as_mov_scale = 1.0    # optional scaling to map prior magnitudes to MOV units if needed

g_eff = {t: games_played[t] + prior_weight for t in teams_list}

# We'll build rows dynamically, then stack at the end so we can easily append priors.
rows = []
targets = []

# --- Real games (row-weighted) ---
for _, row in games.iterrows():
    home_team = row['HomeTeam']
    away_team = row['AwayTeam']
    home_pts  = row['HomePoints']
    away_pts  = row['AwayPoints']
    n
    i = team_to_idx[home_team]
    j = team_to_idx[away_team]

    # rating_i - rating_j = encoded(MOV)
    eq = np.zeros(n, dtype=float)
    eq[i] = 1.0
    eq[j] = -1.0

    mov = home_pts - away_pts
    target = encode_mov(mov, method='capped_with_bonus', cap = 28, winners_bonus= winners_bonus)  # <-- same encoder used for priors below

    # Row weight to normalize leverage by games played
    # w_ij = sqrt( 2 / (g_eff[i] + g_eff[j]) )
    w = np.sqrt(2.0 / (g_eff[home_team] + g_eff[away_team]))

    rows.append(eq * w)
    targets.append(target * w)

# --- Priors as "Week 0" pseudo-games vs neutral baseline (rating = 0) ---
# Same encoder; keep sqrt(prior_weight) so prior acts like N games, then apply the same normalization idea with g_eff(neutral)=0.
scale_prior = np.sqrt(prior_weight)

for team, prior_rating in prior_dict.items():
    idx = team_to_idx.get(team)
    if idx is None:
        continue  # skip priors for teams not in the current team set

    eq_prior = np.zeros(n, dtype=float)
    eq_prior[idx] = 1.0  # (team) - (neutral=0)

    mov_prior = prior_as_mov_scale * prior_rating
    target_prior = encode_mov(mov_prior, method='capped_with_bonus', cap = 100, winners_bonus= winners_bonus)

    # For the normalization, treat opponent (neutral) as g_eff = 0:
    # w_i0 = sqrt( 2 / (g_eff[i] + 0) ) = sqrt( 2 / g_eff[i] )
    w_prior = np.sqrt(2.0 / (g_eff[team]))

    # Combine: first weight as "prior_weight games" (scale_prior), then normalize row like real games (w_prior)
    rows.append(eq_prior * (scale_prior * w_prior))
    targets.append(target_prior * (scale_prior * w_prior))

# --- Sum-of-ratings = 0 constraint to anchor the system ---
sum_constraint = np.ones(n, dtype=float)
rows.append(sum_constraint)
targets.append(0.0)

# Stack into matrix/vector
M = np.vstack(rows)
y = np.array(targets, dtype=float)

# Solve in least squares sense (M @ r ~ y)
massey_ratings, residuals, rank, s = np.linalg.lstsq(M, y, rcond=None)

massey_df = pd.DataFrame({
    'Team': teams_list,
    'MasseyRating': massey_ratings
}).sort_values('MasseyRating', ascending=False)


In [21]:


pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
massey_df["Rank"] = range(1, len(massey_df) + 1)
massey_df.to_csv('rankings.csv')
# Example DataFrame

# Print full DataFrame
print(massey_df.reset_index(drop=True))

                                  Team  MasseyRating  Rank
0                              Indiana     73.405927     1
1                           Ohio State     72.135773     2
2                           Notre Dame     70.530710     3
3                               Oregon     70.360847     4
4                           Texas Tech     69.569254     5
5                                Miami     66.469148     6
6                                 Utah     66.463010     7
7                              Georgia     64.348455     8
8                           Vanderbilt     62.604512     9
9                             Oklahoma     62.602705    10
10                                 BYU     62.581938    11
11                           Texas A&M     62.277596    12
12                                 USC     61.357894    13
13                             Alabama     61.225864    14
14                            Ole Miss     61.155844    15
15                          Washington     59.871167    

In [123]:
# Add a new cell to generate betting lines for unplayed games using the existing `massey_df`.

# Find all unplayed games (not completed, regular season)
remaining_games = data[
    (data["Completed"] == False) & (data["SeasonType"] != 'postseason')
][["HomeTeam", "AwayTeam", "Week"]].copy()

# Merge ratings for home/away
team_ratings = dict(zip(massey_df["Team"], massey_df["MasseyRating"]))

def get_line(home, away):
    home_rating = team_ratings.get(home, 0)
    away_rating = team_ratings.get(away, 0)
    diff = home_rating - away_rating
    # Round to nearest 0.5 like betting lines
    spread = round(diff * 2) / 2.0
    if spread > 0:
        line = f"{home} -{abs(spread)}"
    elif spread < 0:
        line = f"{away} -{abs(spread)}"
    else:
        line = "Pick'em"
    return line, spread

remaining_games["Line"], remaining_games["Spread"] = zip(
    *remaining_games.apply(lambda row: get_line(row["HomeTeam"], row["AwayTeam"]), axis=1)
)

# Save to CSV
output_path = "cfb_betting_lines.csv"
remaining_games.to_csv(output_path, index=False)

remaining_games[remaining_games["Week"]==11]


,HomeTeam,AwayTeam,Week,Line,Spread
2857,Akron,Massachusetts,11,Akron -11.5,11.5
2858,Ohio,Miami (OH),11,Ohio -5.0,5.0
2859,Toledo,Northern Illinois,11,Toledo -11.5,11.5
2860,Ball State,Kent State,11,Kent State -3.0,-3.0
2861,App State,Georgia Southern,11,App State -1.0,1.0
2862,South Florida,UTSA,11,South Florida -12.0,12.0
2863,Morgan State,Delaware State,11,Delaware State -9.5,-9.5
2864,Columbia,Harvard,11,Harvard -35.5,-35.5
2865,UCF,Houston,11,Houston -10.5,-10.5
2866,USC,Northwestern,11,USC -14.5,14.5


In [124]:
# Ensure we have a copy (avoid chained assignment issues)
filtered_data = filtered_data.copy()
games = games.copy()

# Coerce points to numeric and drop any rows with missing points
for col in ["HomePoints", "AwayPoints"]:
    games[col] = pd.to_numeric(games[col], errors="coerce")

bad_pts = games[games[["HomePoints","AwayPoints"]].isna().any(axis=1)]
print(f"Games with missing points: {len(bad_pts)}")
print(bad_pts.head(10))

# Optional: if any slipped through, drop them
games = games.dropna(subset=["HomePoints", "AwayPoints"])

# Double-check Completed truly lines up with points
weird_completed = filtered_data[
    (filtered_data["Completed"] == True) &
    (filtered_data[["HomePoints","AwayPoints"]].isna().any(axis=1))
]
print(f"'Completed==True' but missing points: {len(weird_completed)}")


Games with missing points: 0
Empty DataFrame
Columns: [HomeTeam, AwayTeam, HomePoints, AwayPoints, Score Differential]
Index: []
'Completed==True' but missing points: 0
